# Capítulo 5 — Métodos Numéricos em Finanças

Este capítulo transforma problemas financeiros em algoritmos numéricos. A sequência conceitual é guiada pelo Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander, sem reproduzir o texto da obra.

**Objetivos:**

- resolver raízes, interpolar curvas e otimizar funções;
- estimar volatilidade implícita por bisseção e Newton;
- calcular Greeks, aproximar PDEs e precificar por árvores;
- usar Monte Carlo com medida risk-neutral, seed, erro padrão e convergência;
- simular Normal e Student-t multivariadas com Cholesky.

Em cada método distinguimos estabilidade, velocidade, hipóteses e interpretação financeira.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import interpolate, optimize, stats
from scipy.stats import norm

from quantfinance.numerical import (
    bisection,
    cubic_spline_curve,
    linear_interpolation,
    newton_raphson,
)
from quantfinance.options import (
    black_scholes_call,
    binomial_american,
    binomial_european,
    finite_difference_delta,
    finite_difference_gamma,
    implied_volatility_bisection,
    implied_volatility_newton,
    monte_carlo_european_option,
)
from quantfinance.simulation import (
    simulate_correlated_normal,
    simulate_multivariate_student_t,
)

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 5.1 — Bisseção e Newton-Raphson

Queremos resolver $f(x)=0$. A bisseção exige um intervalo $[a,b]$ com sinais opostos e reduz o intervalo pela metade a cada passo. É lenta, mas robusta.

Newton-Raphson usa uma aproximação local:

$$x_{k+1}=x_k-\frac{f(x_k)}{f'(x_k)}.$$

É rápida perto da raiz, mas depende de um bom ponto inicial e de uma derivada não muito pequena.

**Aplicação financeira:** raízes aparecem em break-even, taxas internas de retorno e volatilidade implícita.

**Exercício:** altere o ponto inicial de Newton e observe quando ele sai da bacia de convergência.

In [ ]:
def root_function(value):
    return value**3 - 2 * value - 5


def root_derivative(value):
    return 3 * value**2 - 2

root_bisect = bisection(root_function, 2.0, 3.0)
root_newton = newton_raphson(root_function, root_derivative, 2.5)
print("Raiz por bisseção:", root_bisect)
print("Raiz por Newton-Raphson:", root_newton)

bisection_widths = [1.0 / (2**iteration) for iteration in range(1, 12)]
newton_errors = []
value = 2.5
for _ in range(6):
    value -= root_function(value) / root_derivative(value)
    newton_errors.append(abs(root_function(value)))

plt.figure(figsize=(8, 4))
plt.semilogy(range(1, 13), [abs(root_function(root_bisect)) + width for width in bisection_widths] + [abs(root_function(root_bisect))], label="bisseção: largura")
plt.semilogy(range(1, 7), newton_errors, "o-", label="Newton: erro residual")
plt.title("Robustez versus velocidade de convergência")
plt.legend()
plt.show()

## 5.2 — Volatilidade implícita: bisseção versus Newton

Para uma call europeia, a volatilidade implícita resolve

$$BS(\sigma)-P_{mercado}=0.$$

A bisseção é robusta quando encontramos um bracket com sinais opostos. Newton é mais rápido, mas precisa da Vega:

$$\sigma_{k+1}=\sigma_k-\frac{BS(\sigma_k)-P_{mercado}}{\operatorname{Vega}(\sigma_k)}.$$

Perto de Vega muito baixa, pequenos erros de preço geram grandes mudanças de volatilidade; Newton pode dar passos inválidos ou divergir. A bisseção não precisa da Vega, mas converge linearmente.

**Interpretação financeira:** a volatilidade implícita é um parâmetro que reproduz um preço observado sob Black-Scholes; não é uma volatilidade histórica nem uma probabilidade subjetiva.

**Exercício:** repita com uma opção muito fora do dinheiro e compare a robustez dos métodos.

In [ ]:
spot, strike, maturity, rate, true_volatility = 100.0, 100.0, 1.0, 0.05, 0.30
market_price = black_scholes_call(spot, strike, maturity, rate, true_volatility)
vol_bisection = implied_volatility_bisection(market_price, spot, strike, maturity, rate)
vol_newton = implied_volatility_newton(market_price, spot, strike, maturity, rate)

print("Preço de mercado simulado:", market_price)
print("Volatilidade verdadeira:", true_volatility)
print("Volatilidade por bisseção:", vol_bisection)
print("Volatilidade por Newton:", vol_newton)
print("Erro bisseção:", vol_bisection - true_volatility)
print("Erro Newton:", vol_newton - true_volatility)
print("Vega no ponto verdadeiro:", spot * norm.pdf(
    (np.log(spot / strike) + (rate + 0.5 * true_volatility**2) * maturity)
    / (true_volatility * np.sqrt(maturity))
) * np.sqrt(maturity))

## 5.3 — Interpolação linear, bilinear, polinomial e splines

Interpolação estima valores entre pontos observados. A linear é local e estável; a polinomial usa um único polinômio e pode oscilar; a spline cúbica usa polinômios locais suaves.

A interpolação bilinear aplica duas interpolações lineares sucessivas em uma grade $(x,y)$, útil para superfícies de volatilidade e preços por strike/maturidade.

**Aplicação financeira:** curvas de juros, superfícies de volatilidade e tabelas de preços precisam de valores entre nós observados.

**Exercício:** compare o erro de interpolação linear, polinomial e spline em uma função conhecida.

In [ ]:
x_nodes = np.array([1, 2, 3, 5, 7, 10], dtype=float)
y_nodes = np.array([0.08, 0.085, 0.09, 0.095, 0.097, 0.10])
grid = np.linspace(1, 10, 200)

linear_values = linear_interpolation(x_nodes, y_nodes, grid)
cubic_values = cubic_spline_curve(x_nodes, y_nodes, grid)
polynomial = np.poly1d(np.polyfit(x_nodes, y_nodes, deg=3))
polynomial_values = polynomial(grid)

# Interpolação bilinear em uma pequena superfície strike/maturidade.
strike_grid = np.array([90.0, 100.0])
maturity_grid = np.array([0.5, 1.0])
price_grid = np.array([[14.0, 10.0], [18.0, 14.0]])
strike_point, maturity_point = 95.0, 0.75
weight_strike = (strike_point - strike_grid[0]) / np.diff(strike_grid)[0]
weight_maturity = (maturity_point - maturity_grid[0]) / np.diff(maturity_grid)[0]
bilinear_value = (
    (1 - weight_strike) * (1 - weight_maturity) * price_grid[0, 0]
    + weight_strike * (1 - weight_maturity) * price_grid[0, 1]
    + (1 - weight_strike) * weight_maturity * price_grid[1, 0]
    + weight_strike * weight_maturity * price_grid[1, 1]
)

print("Preço interpolado bilinearmente:", bilinear_value)
plt.figure(figsize=(9, 4))
plt.plot(x_nodes, y_nodes, "o", label="nós")
plt.plot(grid, linear_values, label="linear")
plt.plot(grid, polynomial_values, label="polinomial")
plt.plot(grid, cubic_values, label="spline cúbica")
plt.title("Interpolação de uma curva financeira")
plt.legend()
plt.show()

## 5.4 — Gradiente, otimização, least squares, likelihood e EM

Métodos de gradiente atualizam parâmetros na direção de redução do objetivo:

$$\theta_{k+1}=\theta_k-\eta\nabla L(\theta_k).$$

Least squares minimiza resíduos quadráticos; maximum likelihood maximiza a probabilidade dos dados. Em uma mistura Normal, o algoritmo EM alterna responsabilidades (E-step) e atualização dos parâmetros (M-step).

**Interpretação financeira:** otimização aparece em calibração, ajuste de curvas, estimação de fatores e calibração de modelos de volatilidade.

**Exercício:** mude a taxa de aprendizado e observe a diferença entre convergência lenta e instável.

In [ ]:
def objective(values):
    x_value, y_value = values
    return (x_value - 2.0) ** 2 + 2.0 * (y_value + 1.0) ** 2

optimization = optimize.minimize(objective, x0=[0.0, 0.0], method="BFGS")
print("Otimização por gradiente:", optimization.x)
print("Objetivo mínimo:", optimization.fun)

# Least squares e MLE Normal em uma amostra financeira simulada.
rng = np.random.default_rng(123)
regressor = rng.normal(size=300)
target = 0.5 + 1.2 * regressor + rng.normal(scale=0.5, size=300)
least_squares = optimize.least_squares(
    lambda parameters: parameters[0] + parameters[1] * regressor - target,
    x0=[0.0, 0.0],
)
print("Least squares alpha/beta:", least_squares.x)

sample = rng.normal(0.002, 0.015, 1_000)
normal_mean_mle = sample.mean()
normal_std_mle = sample.std(ddof=0)
print("MLE Normal analítico:", normal_mean_mle, normal_std_mle)

### EM para uma mistura Normal

Uma mistura representa regimes, por exemplo baixa e alta volatilidade. No E-step calculamos a probabilidade de cada observação pertencer a cada regime; no M-step atualizamos pesos, médias e variâncias.

In [ ]:
mixture_sample = np.concatenate([
    rng.normal(-0.01, 0.01, 500),
    rng.normal(0.015, 0.025, 500),
])
weights = np.array([0.5, 0.5])
means = np.array([-0.005, 0.01])
variances = np.array([0.0004, 0.0009])
for _ in range(30):
    densities = np.column_stack([
        weights[index] * norm.pdf(mixture_sample, means[index], np.sqrt(variances[index]))
        for index in range(2)
    ])
    responsibilities = densities / densities.sum(axis=1, keepdims=True)
    effective_counts = responsibilities.sum(axis=0)
    weights = effective_counts / mixture_sample.size
    means = (responsibilities * mixture_sample[:, None]).sum(axis=0) / effective_counts
    variances = (responsibilities * (mixture_sample[:, None] - means) ** 2).sum(axis=0) / effective_counts
print("EM pesos:", weights)
print("EM médias:", means)
print("EM desvios padrão:", np.sqrt(variances))

## 5.5 — Diferenças finitas, Greeks e PDEs

Diferenças finitas aproximam derivadas com valores vizinhos:

$$\Delta\approx\frac{V(S+h)-V(S-h)}{2h}, \qquad
\Gamma\approx\frac{V(S+h)-2V(S)+V(S-h)}{h^2}.$$

A equação de Black-Scholes para uma opção europeia é

$$\frac{\partial V}{\partial t}+\frac{1}{2}\sigma^2S^2\frac{\partial^2V}{\partial S^2}+rS\frac{\partial V}{\partial S}-rV=0.$$

Uma malha de diferenças finitas aproxima derivadas no espaço e no tempo, propagando o payoff terminal para trás.

**Interpretação financeira:** delta e gamma são sensibilidades locais; a PDE impõe consistência entre tempo, preço, volatilidade e desconto risk-neutral.

**Exercício:** reduza o passo $h$ e observe o trade-off entre erro de truncamento e erro numérico.

In [ ]:
spot, strike, maturity, rate, volatility = 100.0, 100.0, 1.0, 0.05, 0.20
print("Delta numérico:", finite_difference_delta(spot, strike, maturity, rate, volatility))
print("Gamma numérico:", finite_difference_gamma(spot, strike, maturity, rate, volatility))

# Esquema explícito simples para a PDE de uma call europeia.
space_steps, time_steps = 120, 1_200
maximum_spot = 3 * strike
spot_grid = np.linspace(0, maximum_spot, space_steps + 1)
dt = maturity / time_steps
dS = maximum_spot / space_steps
values = np.maximum(spot_grid - strike, 0.0)
for time_index in range(time_steps - 1, -1, -1):
    interior = np.arange(1, space_steps)
    delta = (values[interior + 1] - values[interior - 1]) / (2 * dS)
    gamma = (values[interior + 1] - 2 * values[interior] + values[interior - 1]) / dS**2
    values[interior] += dt * (
        0.5 * volatility**2 * spot_grid[interior]**2 * gamma
        + rate * spot_grid[interior] * delta
        - rate * values[interior]
    )
    values[0] = 0.0
    values[-1] = maximum_spot - strike * np.exp(-rate * (maturity - time_index * dt))
pde_price = np.interp(spot, spot_grid, values)
print("Preço PDE explícita:", pde_price)
print("Preço Black-Scholes:", black_scholes_call(spot, strike, maturity, rate, volatility))

## 5.6 — Árvore binomial, arbitragem e valuation risk-neutral

No modelo Cox-Ross-Rubinstein:

$$u=e^{\sigma\sqrt{\Delta t}}, \qquad d=\frac{1}{u},$$

$$p=\frac{e^{r\Delta t}-d}{u-d}.$$

O preço esperado sob a medida risk-neutral é descontado à taxa livre de risco. A backward induction calcula

$$V_t=e^{-r\Delta t}\left[pV_{up}+(1-p)V_{down}\right].$$

Para uma opção americana:

$$V_t=\max(\text{valor intrínseco},\text{valor de continuação}).$$

Se $p$ está fora de $[0,1]$, a árvore não representa uma medida risk-neutral válida com esses parâmetros; isso é um sinal de arbitragem/inconsistência discretizada.

**Importante:** a probabilidade risk-neutral é uma medida de precificação que reproduz ausência de arbitragem, não a probabilidade subjetiva de o ativo subir.

**Exercício:** aumente $N$ e compare a call europeia binomial com Black-Scholes.

In [ ]:
def tree_parameters(spot, maturity, rate, volatility, steps):
    dt = maturity / steps
    up = np.exp(volatility * np.sqrt(dt))
    down = 1.0 / up
    probability = (np.exp(rate * dt) - down) / (up - down)
    return dt, up, down, probability


dt, up, down, probability = tree_parameters(100.0, 1.0, 0.05, 0.20, 100)
print("dt, u, d, p:", dt, up, down, probability)

steps_grid = np.array([25, 50, 100, 200, 400])
european_prices = np.array([
    binomial_european(100, 100, 1, 0.05, 0.20, steps=int(steps), option="call")
    for steps in steps_grid
])
american_put = binomial_american(100, 100, 1, 0.05, 0.20, steps=400, option="put")
black_scholes_price = black_scholes_call(100, 100, 1, 0.05, 0.20)

print("Call Black-Scholes:", black_scholes_price)
print("Calls binomiais:", european_prices)
print("Put americana:", american_put)

plt.figure(figsize=(8, 4))
plt.plot(steps_grid, european_prices, "o-", label="binomial europeu")
plt.axhline(black_scholes_price, color="black", linestyle="--", label="Black-Scholes")
plt.xlabel("Número de passos N")
plt.ylabel("Preço")
plt.legend()
plt.title("Convergência da árvore binomial")
plt.show()

## 5.7 — Monte Carlo, pseudoaleatoriedade e simulação correlacionada

Sob a medida risk-neutral, o terminal do GBM é

$$S_T=S_0\exp\left[\left(r-\frac{1}{2}\sigma^2\right)T+\sigma\sqrt{T}Z\right], \qquad Z\sim N(0,1).$$

O preço é a média descontada dos payoffs. O erro padrão da média é

$$SE=\frac{s_{payoff}}{\sqrt{M}}.$$

A convergência é tipicamente de ordem $M^{-1/2}$, mais lenta que a árvore em problemas simples, mas Monte Carlo escala melhor para muitas dimensões. Seeds tornam os exemplos reprodutíveis.

Para simular choques correlacionados, usamos $Z=L\varepsilon$, com $LL'=C$. Para Student-t multivariada, dividimos o vetor Normal correlacionado pela raiz de uma variável qui-quadrado escalada.

**Interpretação financeira:** Monte Carlo estima um valor esperado sob a medida de precificação escolhida; o erro padrão mede incerteza amostral, não risco de mercado.

**Exercício:** trace o erro padrão em função de $M$ em escala log-log e compare com a taxa $M^{-1/2}$.

In [ ]:
simulation_sizes = np.array([1_000, 5_000, 20_000, 100_000])
monte_carlo_results = [
    monte_carlo_european_option(
        100.0, 100.0, 1.0, 0.05, 0.20,
        simulations=int(size), seed=123, option="call",
    )
    for size in simulation_sizes
]

for size, result in zip(simulation_sizes, monte_carlo_results):
    print(size, "preço:", result.price, "erro padrão:", result.standard_error)

plt.figure(figsize=(8, 4))
plt.loglog(simulation_sizes, [result.standard_error for result in monte_carlo_results], "o-", label="erro Monte Carlo")
plt.loglog(simulation_sizes, monte_carlo_results[0].standard_error * np.sqrt(simulation_sizes[0] / simulation_sizes), "--", label="$M^{-1/2}$")
plt.xlabel("Número de simulações")
plt.ylabel("Erro padrão")
plt.legend()
plt.title("Convergência de Monte Carlo")
plt.show()

### Monte Carlo com correlação via Cholesky

In [ ]:
correlation = np.array([[1.0, 0.6], [0.6, 1.0]])
correlated_normal = simulate_correlated_normal(correlation, 100_000, seed=123)
print("Correlação empírica Normal:")
print(np.corrcoef(correlated_normal, rowvar=False))

### Simulação multivariada Student-t

In [ ]:
student_t_multivariate = simulate_multivariate_student_t(
    correlation, degrees_of_freedom=5.0, samples=100_000, seed=123
)
print("Correlação empírica Student-t multivariada:")
print(np.corrcoef(student_t_multivariate, rowvar=False))
print("Curtose amostral da primeira margem:", stats.kurtosis(student_t_multivariate[:, 0], fisher=True))

## Exercícios integradores

1. Compare bisseção e Newton em uma raiz com diferentes pontos iniciais.
2. Calcule volatilidade implícita por ambos os métodos e investigue uma Vega baixa.
3. Compare interpolação linear, polinomial, bilinear e spline em uma curva ou superfície.
4. Faça uma descida de gradiente simples e compare com `scipy.optimize`.
5. Implemente mais iterações de EM e acompanhe a log-likelihood da mistura.
6. Compare Greeks por diferenças finitas com valores analíticos.
7. Compare a solução de uma PDE com Black-Scholes e avalie o erro da malha.
8. Verifique se uma árvore produz probabilidade risk-neutral em $[0,1]$.
9. Compare call europeia binomial, Black-Scholes e Monte Carlo.
10. Explique por que a medida risk-neutral não deve ser interpretada como previsão subjetiva.
11. Compare correlação e caudas da Normal multivariada e da Student-t multivariada.